# Dimensional Model – Dimensions

Notebook responsável pela criação das **dimensões analíticas** do modelo dimensional (Adventure Works)

Inclui dimensões com **chaves substitutas**, **SCD2** e documentação no Unity Catalog.


In [0]:
#Imports e setup

from pyspark.sql import functions as F
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import xxhash64
from datetime import datetime

In [0]:
# Leitura das tabelas Silver (clean)

df_customer        = spark.table("adventure_works_catalog.silver.clean_sales_customer")
df_person          = spark.table("adventure_works_catalog.silver.clean_person")
df_store           = spark.table("adventure_works_catalog.silver.clean_sales_store")
df_inventory       = spark.table("adventure_works_catalog.silver.clean_production_inventory")
df_person_address  = spark.table("adventure_works_catalog.silver.clean_person_address")
df_vendor          = spark.table("adventure_works_catalog.silver.clean_purchasing_vendor")
df_product         = spark.table("adventure_works_catalog.silver.clean_production")
df_currency        = spark.table("adventure_works_catalog.silver.clean_sales_currency")
df_currency_rate   = spark.table("adventure_works_catalog.silver.clean_sales_currency_rate")


# Criação de coluna para comentários e documentação
def add_column_comments(catalog, schema, table, columns_dict):
    for column, comment in columns_dict.items():
        spark.sql(f"""
            ALTER TABLE `{catalog}`.`{schema}`.`{table}`
            ALTER COLUMN `{column}`
            COMMENT '{comment}'
        """)


In [0]:
# ============================================================
# DIM_PRODUCT — SILVER 
# ============================================================


df_product_base = (
    df_product
    .select(
        F.col("ProductID"),                # NK
        F.col("ProductName"),
        F.col("ProductCategoryName").alias("Category"),
        F.col("ProductSubcategoryName").alias("SubCategory"),
        F.col("ProductModelName").alias("ModelName"),
        F.col("StandardCost"),
        F.col("ListPrice")
    )
)


# Verificar se a DIM_PRODUCT já existe na SILVER
if spark.catalog.tableExists("adventure_works_catalog.silver.dim_product"):
    df_silver_current = (
        spark.read.table("adventure_works_catalog.silver.dim_product")
        .filter(F.col("IsCurrent") == True)
    )
else:
    df_silver_current = None


# Detectar novos produtos ou mudanças (CORE SCD2)
if df_silver_current is not None:
    df_changes = (
        df_product_base.alias("src")
        .join(
            df_silver_current.alias("tgt"),
            on="ProductID",
            how="left"
        )
        .where(
            F.col("tgt.ProductID").isNull() |
            (F.col("src.ProductName")  != F.col("tgt.ProductName")) |
            (F.col("src.Category")     != F.col("tgt.Category")) |
            (F.col("src.SubCategory")  != F.col("tgt.SubCategory")) |
            (F.col("src.ModelName")    != F.col("tgt.ModelName")) |
            (F.col("src.StandardCost") != F.col("tgt.StandardCost")) |
            (F.col("src.ListPrice")    != F.col("tgt.ListPrice"))
        )
        .select("src.*")  
    )
else:
    df_changes = df_product_base


# Criar novas versões (SK + vigência)
df_new_versions = (
    df_changes
    .withColumn("StartDate", F.current_date())
    .withColumn("EndDate", F.lit("9999-12-31").cast("date"))
    .withColumn("IsCurrent", F.lit(True))
    .withColumn(
        "Product_SK",
        xxhash64(
            "ProductID",
            "ProductName",
            "Category",
            "SubCategory",
            "ModelName",
            "StandardCost",
            "ListPrice",
            "StartDate"
        )
    )
)


# Fechar versões antigas
if df_silver_current is not None:
    df_closed = (
        df_silver_current
        .join(
            df_changes.select("ProductID"),
            on="ProductID",
            how="inner"
        )
        .withColumn("EndDate", F.current_date())
        .withColumn("IsCurrent", F.lit(False))
    )

    df_dim_product_silver = (
        df_silver_current
        .join(
            df_closed.select("Product_SK"),
            on="Product_SK",
            how="left_anti"
        )
        .unionByName(df_closed)
        .unionByName(df_new_versions)
    )
else:
    df_dim_product_silver = df_new_versions


#  Seleção final 
df_dim_product_silver = (
    df_dim_product_silver
    .select(
        "Product_SK",
        "ProductID",
        "ProductName",
        "Category",
        "SubCategory",
        "ModelName",
        "StandardCost",
        "ListPrice",
        "StartDate",
        "EndDate",
        "IsCurrent"
    )
)


#  Validação
df_dim_product_silver.printSchema()
df_dim_product_silver.display()


# Escrita na SILVER (SCD2)
(
    df_dim_product_silver.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("adventure_works_catalog.silver.dim_product")
)

# Descrição das colunas
dim_product_columns = {
    "Product_SK": "Chave substituta da versão do produto (SCD2)",
    "ProductID": "Identificador natural do produto",
    "ProductName": "Nome do produto",
    "Category": "Categoria do produto",
    "SubCategory": "Subcategoria do produto",
    "ModelName": "Modelo do produto",
    "StandardCost": "Custo padrão do produto",
    "ListPrice": "Preço de lista do produto",
    "StartDate": "Data de início da vigência da versão",
    "EndDate": "Data de fim da vigência da versão",
    "IsCurrent": "Indicador da versão atual do produto"
}

# Adicionando os comentários no Unity Catalog
add_column_comments(
    catalog="adventure_works_catalog",
    schema="silver",
    table="dim_product",
    columns_dict=dim_product_columns
)


In [0]:
# =====================================================
# DIM_CUSTOMER - SILVER
# =====================================================


df_dim_customer_base = (
    df_customer.alias("c")

    .join(
        df_person.alias("p"),
        F.col("c.PersonID") == F.col("p.PersonID"),
        "left"
    )

    .join(
        df_store.alias("s"),
        F.col("c.StoreID") == F.col("s.StoreID"),
        "left"
    )

    .join(
        df_territory.alias("t"),
        F.col("c.TerritoryID") == F.col("t.TerritoryID"),
        "left"
    )

    .select(
        # Chave natural
        F.col("c.CustomerID"),

        # Identificação (NULL quando não aplicável)
        F.when(
            F.col("c.PersonID").isNotNull(),
            F.col("c.PersonID")
        ).otherwise(F.lit(None)).alias("PersonID"),

        F.col("c.StoreID"),

        # Tipo de cliente
        F.when(
            F.col("c.PersonID").isNotNull(),
            F.lit("PERSON")
        ).otherwise(F.lit("STORE")).alias("CustomerType"),

        # Nome da pessoa (NULL quando STORE)
        F.when(
            F.col("c.PersonID").isNotNull(),
            F.concat_ws(" ", F.col("p.FirstName"), F.col("p.LastName"))
        ).otherwise(F.lit(None)).alias("PersonName"),

        # Nome da loja (NULL quando PERSON)
        F.when(
            F.col("c.StoreID").isNotNull(),
            F.col("s.StoreName")
        ).otherwise(F.lit(None)).alias("StoreName"),

        # Território
        F.col("t.TerritoryName"),
        F.col("t.TerritoryGroup")
    )
)

# Garantia de granularidade
df_dim_customer_base = (
    df_dim_customer_base
    .filter(F.col("CustomerID").isNotNull())
    .dropDuplicates(["CustomerID"])
)


# Criação da SK 
df_dim_customer = (
    df_dim_customer_base
    .withColumn(
        "Customer_SK",
        F.row_number().over(Window.orderBy("CustomerID"))
    )
    .select(
        "Customer_SK",        
        "CustomerID",
        "PersonID",
        "StoreID",
        "CustomerType",
        "PersonName",
        "StoreName",
        "TerritoryName",
        "TerritoryGroup"
    )
)


# Escrita na Silver
(
    df_dim_customer.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("adventure_works_catalog.silver.dim_customer")
)

# Descrição das colunas
dim_customer_columns = {
    "Customer_SK": "Chave substituta do cliente",
    "CustomerID": "Identificador natural do cliente",
    "PersonID": "Identificador da pessoa física (NULL para lojas)",
    "StoreID": "Identificador da loja (NULL para pessoas físicas)",
    "CustomerType": "Tipo de cliente: PERSON ou STORE",
    "PersonName": "Nome do cliente quando pessoa física",
    "StoreName": "Nome do cliente quando pessoa jurídica",
    "TerritoryName": "Território de vendas",
    "TerritoryGroup": "Grupo regional do território"
}

# Adicionando os comentários no Unity Catalog
add_column_comments(
    catalog="adventure_works_catalog",
    schema="silver",
    table="dim_customer",
    columns_dict=dim_customer_columns
)


In [0]:
# =====================================================
# DIM_LOCATION - SILVER
# =====================================================


df_location_base = (
    df_person_address
    .select(
        "City",
        "StateProvinceName",
        "CountryName"
    )
    .filter(
        F.col("City").isNotNull() &
        F.col("StateProvinceName").isNotNull() &
        F.col("CountryName").isNotNull()
    )
    .dropDuplicates([
        "City",
        "StateProvinceName",
        "CountryName"
    ])
)


df_dim_location_pre = (
    df_location_base.alias("loc")
    .join(
        df_sales_territory.alias("t"),
        F.col("loc.CountryName") == F.col("t.CountryRegionCode"),
        "left"
    )
    .select(
        F.col("loc.City"),
        F.col("loc.StateProvinceName"),
        F.col("loc.CountryName"),
        F.col("t.TerritoryName")
    )
)


# Criação da Surrogate Key
window_spec = Window.orderBy(
    "CountryName",
    "StateProvinceName",
    "City"
)

df_dim_location = (
    df_dim_location_pre
    .withColumn(
        "Location_SK",
        F.row_number().over(window_spec)
    )
    .select(
        "Location_SK",
        "City",
        "StateProvinceName",
        "CountryName",
        "TerritoryName"
    )
)


# Conferência
df_dim_location.printSchema()
df_dim_location.display()


# Escrita na Gold
(
    df_dim_location.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "adventure_works_catalog.silver.dim_location"
    )
)

# Descrição das colunas
dim_location_columns = {
    "Location_SK": "Chave substituta da localização",
    "City": "Cidade da localização",
    "StateProvinceName": "Estado ou província",
    "CountryName": "País",
    "TerritoryName": "Território de vendas associado ao país"
}


# Adicionando os comentários no Unity Catalog
add_column_comments(
    catalog="adventure_works_catalog",
    schema="silver",
    table="dim_location",
    columns_dict=dim_location_columns
)



In [0]:
# =====================================================
# DIM_DATE - SILVER
# =====================================================


# Definição do intervalo de datas
start_date = "2010-01-01"
end_date   = "2025-12-31"

# Geração da sequência de datas
df_date = (
    spark
    .range(1)
    .select(
        F.explode(
            F.sequence(
                F.to_date(F.lit(start_date)),
                F.to_date(F.lit(end_date)),
                F.expr("interval 1 day")
            )
        ).alias("FullDate")
    )
)


df_dim_date = (
    df_date
    .select(
        # Chave da data (yyyyMMdd)
        F.date_format("FullDate", "yyyyMMdd").cast("int").alias("DateKey"),

        F.col("FullDate"),

        # Dia
        F.dayofweek("FullDate").alias("DayNumberOfWeek"),
        F.date_format("FullDate", "EEEE").alias("DayNameOfWeek"),
        F.dayofmonth("FullDate").alias("DayNumberOfMonth"),

        # Mês
        F.month("FullDate").alias("MonthNumberOfYear"),
        F.date_format("FullDate", "MMMM").alias("MonthName"),

        # Trimestre / Ano
        F.quarter("FullDate").alias("CalendarQuarter"),
        F.year("FullDate").alias("CalendarYear"),

        # Final de semana
        F.when(
            F.dayofweek("FullDate").isin([1, 7]),
            F.lit(1)
        ).otherwise(F.lit(0)).alias("IsWeekend")
    )
)

# Garantir unicidade
df_dim_date = df_dim_date.dropDuplicates(["DateKey"])

# Validação
df_dim_date.printSchema()

(
    df_dim_date.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(
        "adventure_works_catalog.silver.dim_date"
    )
)

# Descrição das colunas
dim_date_columns = {
    "DateKey": "Chave substituta da data no formato YYYYMMDD",
    "FullDate": "Data completa no formato calendário",
    "DayNumberOfWeek": "Número do dia da semana (1 = domingo, 7 = sábado)",
    "DayNameOfWeek": "Nome do dia da semana",
    "DayNumberOfMonth": "Número do dia no mês",
    "MonthNumberOfYear": "Número do mês no ano",
    "MonthName": "Nome do mês",
    "CalendarQuarter": "Trimestre do ano",
    "CalendarYear": "Ano calendário",
    "IsWeekend": "Indicador de final de semana (1 = sim, 0 = não)"
}

# Adicionando os comentários no Unity Catalog
add_column_comments(
    catalog="adventure_works_catalog",
    schema="silver",
    table="dim_date",
    columns_dict=dim_date_columns
)


In [0]:
# =====================================================
# DIM_CURRENCY — SILVER 
# =====================================================

df_dim_currency_pre = (
    df_currency_rate.alias("cr")

    # Join moeda de origem
    .join(
        df_currency.alias("fc"),
        F.col("cr.FromCurrencyCode") == F.col("fc.CurrencyCode"),
        "left"
    )

    # Join moeda de destino
    .join(
        df_currency.alias("tc"),
        F.col("cr.ToCurrencyCode") == F.col("tc.CurrencyCode"),
        "left"
    )

    .select(
        F.col("cr.CurrencyRateID"),
        F.col("cr.FromCurrencyCode"),
        F.col("fc.CurrencyName").alias("FromCurrencyName"),
        F.col("cr.ToCurrencyCode"),
        F.col("tc.CurrencyName").alias("ToCurrencyName"),
        F.col("cr.CurrencyRateDate"),
        F.col("cr.AverageRate"),
        F.col("cr.EndOfDayRate")
    )
)


# Criação da SK
window_spec = Window.orderBy(
    "CurrencyRateDate",
    "FromCurrencyCode",
    "ToCurrencyCode"
)

df_dim_currency = (
    df_dim_currency_pre
    .withColumn(
        "Currency_SK",
        F.row_number().over(window_spec)
    )
    .select(
        "Currency_SK",
        "CurrencyRateID",
        "FromCurrencyCode",
        "FromCurrencyName",
        "ToCurrencyCode",
        "ToCurrencyName",
        "CurrencyRateDate",
        "AverageRate",
        "EndOfDayRate"
    )
)


# Escrita na Silver
(
    df_dim_currency.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "adventure_works_catalog.silver.dim_currency"
    )
)

# Descrição das colunas
dim_currency_columns = {
    "Currency_SK": "Chave substituta da taxa de câmbio",
    "CurrencyRateID": "Identificador natural da taxa de câmbio",
    "FromCurrencyCode": "Código da moeda de origem",
    "FromCurrencyName": "Nome da moeda de origem",
    "ToCurrencyCode": "Código da moeda de destino",
    "ToCurrencyName": "Nome da moeda de destino",
    "CurrencyRateDate": "Data de vigência da taxa de câmbio",
    "AverageRate": "Taxa média de conversão entre as moedas",
    "EndOfDayRate": "Taxa de conversão registrada ao final do dia"
}

# Adicionando os comentários no Unity Catalog
add_column_comments(
    catalog="adventure_works_catalog",
    schema="silver",
    table="dim_currency",
    columns_dict=dim_currency_columns
)



In [0]:
# =====================================================
# DIM_SUPPLIER - SILVER
# =====================================================


df_dim_supplier_pre = (
    df_vendor
    .select(
        F.col("VendorID").alias("SupplierID"),
        F.col("VendorName").alias("SupplierName"),
        F.col("CreditRating").alias("CreditRating"),
        F.col("ActiveFlag").alias("IsActiveSupplier")
    )
    .filter(F.col("SupplierID").isNotNull())
    .dropDuplicates(["SupplierID"])
)


# Criação da Surrogate Key
window_spec = Window.orderBy("SupplierID")

df_dim_supplier = (
    df_dim_supplier_pre
    .withColumn(
        "Supplier_SK",
        F.row_number().over(window_spec)
    )
    .select(
        "Supplier_SK",
        "SupplierID",
        "SupplierName",
        "CreditRating",
        "IsActiveSupplier"
    )
)


# Conferência 
df_dim_supplier.printSchema()
df_dim_supplier.display()


# Escrita na Silver
(
    df_dim_supplier.write
    .format("delta")
    .mode("overwrite")   # SCD Tipo 1
    .saveAsTable(
        "adventure_works_catalog.silver.dim_supplier"
    )
)

# Descrição das colunas
dim_supplier_columns = {
    "Supplier_SK": "Chave substituta do fornecedor",
    "SupplierID": "Identificador natural do fornecedor",
    "SupplierName": "Nome do fornecedor",
    "CreditRating": "Classificação de crédito do fornecedor",
    "IsActiveSupplier": "Indicador se o fornecedor está ativo"
}

# Adicionando comentários no Unity Catalog
add_column_comments(
    catalog="adventure_works_catalog",
    schema="silver",
    table="dim_supplier",
    columns_dict=dim_supplier_columns
)